In [2]:
import xarray as xr
from dask.distributed import Client
from dask_jobqueue import PBSCluster

In [4]:
path = '/glade/work/acruz/Caribbean_Heat_data/ERA5/'
clim_rh = xr.open_dataset(path+'RH_clim_during_HIdmax.nc')
clim_rh

/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


<xarray.Dataset> Size: 478kB
Dimensions:           (month: 12, latitude: 82, longitude: 121)
Coordinates:
  * month             (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude          (latitude) float64 656B 7.75 8.0 8.25 ... 27.5 27.75 28.0
  * longitude         (longitude) float64 968B -89.0 -88.75 ... -59.25 -59.0
Data variables:
    RH_during_HIdmax  (month, latitude, longitude) float32 476kB ...

In [5]:
clim_t = xr.open_dataset(path+'T_clim_during_HIdmax.nc')
clim_t

<xarray.Dataset> Size: 478kB
Dimensions:          (month: 12, latitude: 82, longitude: 121)
Coordinates:
  * month            (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude         (latitude) float64 656B 7.75 8.0 8.25 ... 27.5 27.75 28.0
  * longitude        (longitude) float64 968B -89.0 -88.75 ... -59.25 -59.0
Data variables:
    T_during_HIdmax  (month, latitude, longitude) float32 476kB ...

In [6]:
tds = xr.open_dataset(path+'T_during_peak_HI.nc')
tds

<xarray.Dataset> Size: 1GB
Dimensions:          (time: 31471, latitude: 82, longitude: 121)
Coordinates:
  * time             (time) datetime64[ns] 252kB 1940-01-01 ... 2026-02-28
  * latitude         (latitude) float64 656B 7.75 8.0 8.25 ... 27.5 27.75 28.0
  * longitude        (longitude) float64 968B -89.0 -88.75 ... -59.25 -59.0
Data variables:
    T_during_HIdmax  (time, latitude, longitude) float32 1GB ...

In [7]:
rhds = xr.open_dataset(path+'RH_during_peak_HI.nc')
rhds

<xarray.Dataset> Size: 1GB
Dimensions:           (time: 31471, latitude: 82, longitude: 121)
Coordinates:
  * time              (time) datetime64[ns] 252kB 1940-01-01 ... 2026-02-28
  * latitude          (latitude) float64 656B 7.75 8.0 8.25 ... 27.5 27.75 28.0
  * longitude         (longitude) float64 968B -89.0 -88.75 ... -59.25 -59.0
Data variables:
    RH_during_HIdmax  (time, latitude, longitude) float32 1GB ...

In [12]:
t_anom = tds['T_during_HIdmax'].groupby('time.month') - clim_t
t_anom = t_anom.chunk({'time': -1})
t_anom = t_anom.rename({'T_during_HIdmax': 'Tanom_during_HIdmax'})
t_anom

<xarray.Dataset> Size: 1GB
Dimensions:              (latitude: 82, longitude: 121, time: 31471)
Coordinates:
  * latitude             (latitude) float64 656B 7.75 8.0 8.25 ... 27.75 28.0
  * longitude            (longitude) float64 968B -89.0 -88.75 ... -59.25 -59.0
  * time                 (time) datetime64[ns] 252kB 1940-01-01 ... 2026-02-28
    month                (time) int64 252kB dask.array<chunksize=(31471,), meta=np.ndarray>
Data variables:
    Tanom_during_HIdmax  (time, latitude, longitude) float32 1GB dask.array<chunksize=(31471, 82, 121), meta=np.ndarray>

In [13]:
rh_anom = rhds['RH_during_HIdmax'].groupby('time.month') - clim_rh
rh_anom = rh_anom.chunk({'time': -1}).rename({'RH_during_HIdmax': 'RHanom_during_HIdmax'})
rh_anom

<xarray.Dataset> Size: 1GB
Dimensions:               (latitude: 82, longitude: 121, time: 31471)
Coordinates:
  * latitude              (latitude) float64 656B 7.75 8.0 8.25 ... 27.75 28.0
  * longitude             (longitude) float64 968B -89.0 -88.75 ... -59.25 -59.0
  * time                  (time) datetime64[ns] 252kB 1940-01-01 ... 2026-02-28
    month                 (time) int64 252kB dask.array<chunksize=(31471,), meta=np.ndarray>
Data variables:
    RHanom_during_HIdmax  (time, latitude, longitude) float32 1GB dask.array<chunksize=(31471, 82, 121), meta=np.ndarray>

In [14]:
t_anom.to_netcdf(path+'Tanom_during_HIdmax.nc')

In [17]:
rh_anom.to_netcdf(path+'RHanom_during_HIdmax.nc')

In [19]:
# client.shutdown()